
The goal here is to generate phylogenetic trees for the top 4 families which are represented in each individual's seroreactivity profile.

So I want to build a tree for the Arenaviridae, Nairoviridae, Phenuiviridae and Hantaviridae family.

The origin (represented as a node) should represent the viral family, with branches going out for each genus (represented as a node), ending in leaves (also represented as a circular node) for each species that is represented in the library. The size of each node should correspond to the fraction of seroreactivity observed in the cohort for this family / genus / species.

My input is a peptide dataframe (/data1/sporak/Lassa_fever/Lassa_Fever_growup2/PhIP-Seq/Lassa_Lib/All_source_plates_Analysis/lasv_phipseq_2026/peptides_passing_filtering_082626.csv) which includes the filtered peptides (so hits for this cohort), and this peptide dataframe has two columns which represent the fraction seropositive in this cohort for this peptide, but also columns named family, genus, species which correspond to the taxonomy this peptide belongs to.

Now I am also giving this script the entire peptide library as a peptide metadata (PhIP-Seq/Lassa_Lib/All_source_plates_Analysis/lassa_library_sequences_species_standardized_070726.csv).

here we also have columns representing the family, genus, species.

In [2]:

import math
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
import pandas as pd

hits_df = pd.read_csv('peptides_passing_filtering_090126_1305_ICTV_MSL41_corrected.csv')

long_hits_master_df = pd.read_csv('master_peptide_thresh.csv')

library_df = pd.read_csv('../lassa_library_sequences_species_standardized_070726.csv')

ictv_df = pd.read_csv('ICTV_Master_Species_List_2025_MSL41.v1.csv')

In [4]:
FAMILY_COL = "family"
GENUS_COL = "genus"
SPECIES_COL = "species"

# Fraction of the cohort seropositive to this peptide.
FRACTION_COL = "frac_pass_z_in_SL"

FAMILIES = ["Arenaviridae", "Nairoviridae", "Phenuiviridae", "Hantaviridae", 'Flaviviridae', 'Filoviridae', 'Paramyxoviridae']

# How per-peptide fractions roll up into species -> genus -> family node size:
#   "max"  -> a taxon "lights up" if ANY peptide/species under it is reactive
#   "mean" -> average reactivity across peptides/species under it
NODE_AGG = "max"

OUTDIR = Path("taxonomy_trees_out")
OUTDIR.mkdir(exist_ok=True)

In [5]:
ICTV_FAMILY_COL = "Family"
ICTV_GENUS_COL = "Genus"
ICTV_SPECIES_COL = "Species"


def _norm(s):
    return " ".join(str(s).split())


def load_ictv_species_table():
    ictv = ictv_df 
    ictv = ictv[[ICTV_FAMILY_COL, ICTV_GENUS_COL, ICTV_SPECIES_COL]].dropna()
    for col in (ICTV_FAMILY_COL, ICTV_GENUS_COL, ICTV_SPECIES_COL):
        ictv[col] = ictv[col].map(_norm)
    return ictv.drop_duplicates()


ictv_df = load_ictv_species_table()

In [6]:
# individuals belonging to the "SL" cohort -- the population frac_pass_z_in_SL
# is computed over. Confirm 'S' and 'C' are both part of that SL population
# (vs. HBDB, which is the separate US negative-control cohort) -- adjust if not.
SL_CATEGORIES = ["S", "C"]

# an individual only counts as seropositive to a species if they hit at least
# this many DISTINCT peptides within it -- same idea as GENUS_MIN_PEPTIDES
# in your master script, applied per species rather than per peptide-fraction
MIN_HITS_FOR_INDIVIDUAL_POSITIVE = 3

def species_seroprevalence(species_list_norm, master_df, n_individuals):
    """
    species_list_norm: set of _norm()-ed species names (from ICTV) to restrict to.
    Fraction of the SL cohort seropositive to each species: an individual
    counts as positive only if their per-individual n_hits for that species
    (already computed by level_summary) is >= MIN_HITS_FOR_INDIVIDUAL_POSITIVE.
    """
    sp = master_df[
        (master_df["level"] == "species") & (master_df["category"].isin(SL_CATEGORIES))
    ].copy()
    sp["_species_norm"] = sp["value"].map(_norm)
    sp = sp[sp["_species_norm"].isin(species_list_norm)]
    if sp.empty:
        return pd.Series(dtype=float)

    is_pos = sp["n_hits"] >= MIN_HITS_FOR_INDIVIDUAL_POSITIVE
    n_seropositive = sp.loc[is_pos].groupby("_species_norm")["individual"].nunique()
    return n_seropositive / n_individuals


def build_family_tree(family, master_df, lib_df, ictv_df, n_individuals):
    lib_f = lib_df[lib_df[FAMILY_COL] == family]
    if lib_f.empty:
        print(f"[{family}] not found in library file -- skipping")
        return None

    ictv_f = ictv_df[ictv_df[ICTV_FAMILY_COL] == family]
    if ictv_f.empty:
        print(f"[{family}] not found in ICTV list -- skipping")
        return None
    species_table = ictv_f[[ICTV_GENUS_COL, ICTV_SPECIES_COL]].rename(
        columns={ICTV_GENUS_COL: GENUS_COL, ICTV_SPECIES_COL: SPECIES_COL}
    ).drop_duplicates()

    species_list_norm = set(species_table[SPECIES_COL].map(_norm))
    species_frac = species_seroprevalence(species_list_norm, master_df, n_individuals)

    G = nx.DiGraph()
    G.add_node(family, level="family", size=0.0)

    genus_children = {}
    for _, row in species_table.iterrows():
        genus, species = row[GENUS_COL], row[SPECIES_COL]
        if genus not in G:
            G.add_node(genus, level="genus", size=0.0)
            G.add_edge(family, genus)
        val = float(species_frac.get(_norm(species), 0.0))
        G.add_node(species, level="species", size=val)
        G.add_edge(genus, species)
        genus_children.setdefault(genus, []).append(val)

    for genus, vals in genus_children.items():
        G.nodes[genus]["size"] = max(vals) if NODE_AGG == "max" else sum(vals) / len(vals)

    fam_vals = [G.nodes[g]["size"] for g in genus_children]
    G.nodes[family]["size"] = (
        (max(fam_vals) if NODE_AGG == "max" else sum(fam_vals) / len(fam_vals))
        if fam_vals else 0.0
    )
    return G

In [7]:
import matplotlib.colors as mcolors

_HEATMAP_CMAP = mcolors.LinearSegmentedColormap.from_list("itol_heat", ["#ffffcc", "#800026"])


def _frac_to_hex(val):
    r, g, b, _ = _HEATMAP_CMAP(max(0.0, min(1.0, val)))
    return mcolors.to_hex((r, g, b))


def write_itol_genus_symbols(G, outpath):
    """
    iTOL DATASET_SYMBOL: a filled circle at each genus node, colored on the
    same #ffffcc -> #800026 gradient as the species-level DATASET_HEATMAP.
    Species leaves are untouched -- this is a separate dataset layered on top.
    """
    lines = [
        "DATASET_SYMBOL",
        "SEPARATOR TAB",
        "DATASET_LABEL\tGenus fraction seropositive",
        "COLOR\t#800026",
        "MAXIMUM_SIZE\t15",
        "DATA",
    ]
    for n in G.nodes:
        if G.nodes[n]["level"] != "genus":
            continue
        label = _itol_label(n)
        color = _frac_to_hex(G.nodes[n]["size"])
        # ID  SYMBOL(2=circle)  SIZE  COLOR  FILL(1)  POSITION(1=at node)
        lines.append(f"{label}\t2\t12\t{color}\t1\t1")
    Path(outpath).write_text("\n".join(lines) + "\n")


def _itol_label(n):
    return n.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")


def write_itol_heatmap(G, root, outpath):
    lines = [
        "DATASET_HEATMAP",
        "SEPARATOR TAB",
        "DATASET_LABEL\tFraction seropositive",
        "COLOR\t#800026",
        "COLOR_MIN\t#ffffcc",
        "COLOR_MAX\t#800026",
        "FIELD_LABELS\tfrac_seropositive",
        "STRIP_WIDTH\t60",
        "MARGIN\t15",
        "SHOW_LABELS\t1",
        "LEGEND_TITLE\tFraction seropositive",
        "LEGEND_SHAPES\t1",
        "LEGEND_COLORS\t#ffffcc,#800026",
        "LEGEND_LABELS\t0,1",
        "DATA",
    ]
    for n in G.nodes:
        label = _itol_label(n)
        val = G.nodes[n]["size"]
        lines.append(f"{label}\t{val:.4f}")
    Path(outpath).write_text("\n".join(lines) + "\n")

def to_newick(G, root):
    def rec(n):
        children = list(G.successors(n))
        label = _itol_label(n)
        if not children:
            return label
        return "(" + ",".join(rec(c) for c in children) + ")" + label
    return rec(root) + ";"

In [8]:
N_SL_INDIVIDUALS = long_hits_master_df.loc[
    (long_hits_master_df["level"] == "species")
    & (long_hits_master_df["category"].isin(SL_CATEGORIES)),
    "individual",
].nunique()

trees = {}
for family in FAMILIES:
    G = build_family_tree(family, long_hits_master_df, library_df, ictv_df, N_SL_INDIVIDUALS)
    if G is None:
        continue
    trees[family] = G
    nwk_path = OUTDIR / f"{family}.nwk"
    heatmap_path = OUTDIR / f"{family}_itol_heatmap.txt"
    nwk_path.write_text(to_newick(G, family))
    write_itol_heatmap(G, family, heatmap_path)
    print(f"[{family}] {G.number_of_nodes()} nodes -> {nwk_path}, {heatmap_path}")

    genus_symbol_path = OUTDIR / f"{family}_itol_genus_symbols.txt"
    write_itol_genus_symbols(G, genus_symbol_path)
    print(f"[{family}] ... -> {genus_symbol_path}")

[Arenaviridae] 77 nodes -> taxonomy_trees_out/Arenaviridae.nwk, taxonomy_trees_out/Arenaviridae_itol_heatmap.txt
[Arenaviridae] ... -> taxonomy_trees_out/Arenaviridae_itol_genus_symbols.txt
[Nairoviridae] 69 nodes -> taxonomy_trees_out/Nairoviridae.nwk, taxonomy_trees_out/Nairoviridae_itol_heatmap.txt
[Nairoviridae] ... -> taxonomy_trees_out/Nairoviridae_itol_genus_symbols.txt
[Phenuiviridae] 248 nodes -> taxonomy_trees_out/Phenuiviridae.nwk, taxonomy_trees_out/Phenuiviridae_itol_heatmap.txt
[Phenuiviridae] ... -> taxonomy_trees_out/Phenuiviridae_itol_genus_symbols.txt
[Hantaviridae] 64 nodes -> taxonomy_trees_out/Hantaviridae.nwk, taxonomy_trees_out/Hantaviridae_itol_heatmap.txt
[Hantaviridae] ... -> taxonomy_trees_out/Hantaviridae_itol_genus_symbols.txt
[Flaviviridae] 67 nodes -> taxonomy_trees_out/Flaviviridae.nwk, taxonomy_trees_out/Flaviviridae_itol_heatmap.txt
[Flaviviridae] ... -> taxonomy_trees_out/Flaviviridae_itol_genus_symbols.txt
[Filoviridae] 27 nodes -> taxonomy_trees_out